<a href="https://colab.research.google.com/github/Ophelie4/pfizer-document-intelligence/blob/main/project-05-intro-to-rag/notebooks/Ophelie_RAG_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Build and Optimize a RAG Pipeline for Document Retrieval

**Task:** Build a complete Retrieval-Augmented Generation (RAG) pipeline using LlamaIndex + Gemini over the real reference document `sample-sdf-document (Project 5).pdf` (the same Cytiva ÄKTA ready Flow Kit document used in Project 3). The pipeline covers PDF parsing, chunking, embedding generation, retrieval, and response generation, and is tested on two required prompts.

**Author:** Ophelie


## Step 1: Install dependencies

- `llama-index` — core RAG framework
- `llama-index-embeddings-huggingface` — free, local embedding model
- `llama-index-llms-google-genai` — Gemini LLM integration (see note in Step 6 on why this replaces the deprecated `llama-index-llms-gemini` package from the task instructions)
- `llama-index-retrievers-bm25` — keyword-based retriever for the hybrid retrieval strategy
- `pymupdf` — PDF text extraction
- `gdown` — download the reference PDF directly from its Google Drive file ID
- `nest_asyncio` — patches Colab's running event loop so GoogleGenAI's internal async calls work


In [ ]:
!pip install -q llama-index llama-index-embeddings-huggingface llama-index-llms-google-genai llama-index-retrievers-bm25 pymupdf gdown nest_asyncio google-genai
print("Installation complete.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 57.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.7/74.7 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 83.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 683.3/683.3 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.6/164.6 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 46.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
Installation complete.


## Step 2: Download the real reference document

The task provides a specific, real PDF (not a synthetic document): `sample-sdf-document (Project 5).pdf`, a 3-page Cytiva document covering an ÄKTA ready Flow Kit storage-conditions letter and two Certificates of Quality. It is downloaded directly from its Google Drive file ID.


In [ ]:
import gdown

FILE_ID = "1k_ZRrdSSJLxhMvBnuoaPVYTz05lLmYEN"
url = f"https://drive.google.com/uc?id={FILE_ID}"
output_path = "sample_sdf_document.pdf"

gdown.download(url, output_path, quiet=False)

import os
print("Downloaded:", output_path, "-", os.path.getsize(output_path), "bytes")


Downloading...
From: https://drive.google.com/uc?id=1k_ZRrdSSJLxhMvBnuoaPVYTz05lLmYEN
To: /content/sample_sdf_document.pdf
100%|██████████| 147k/147k [00:00<00:00, 3.95MB/s]

Downloaded: sample_sdf_document.pdf - 147199 bytes


## Step 3: Extract text from the PDF

Using PyMuPDF (`fitz`) to extract raw text from all 3 pages.


In [ ]:
import fitz  # PyMuPDF

doc = fitz.open(output_path)
page_texts = [page.get_text() for page in doc]
full_text = "\n\n".join(page_texts)
doc.close()

print(f"Extracted {len(page_texts)} pages, {len(full_text)} characters total.\n")
print("--- Preview of extracted text (first 800 chars) ---")
print(full_text[:800])


Extracted 3 pages, 4027 characters total.

--- Preview of extracted text (first 800 chars) ---
Cytiva
100 Results Way
Marlborough, MA 01752
United States
Page 1 / 1
cytiva.com
3 June, 2022
Re: ÄKTATM ready Flow Kit Storage Conditions
To Whom It May Concern,
The recommended storage temperature for standard ÄKTA ready flow kits is provided in Section 8.3 of the Operating 
Instructions 28960345 and specified as > +5 C. This recommendation also applies to all modified ÄKTA ready flow kits 
as well, including the two listed in the below table. Extended storage below the recommended +5
could lead to 
brittleness or cracking of the plastic connectors. However, the operating temperature of ÄKTA ready flow kits is +2 C to 
+40 C. If the kits are allowed to acclimate to a warmer temperature before being used this would reduce the risk of
damage to the kit during setup and handling.
Descriptio


## Step 4: Chunking strategy

**Choice: `SentenceSplitter` with `chunk_size=200`, `chunk_overlap=30`.**

**Why:** This document is short (3 pages, ~1,600 characters) and structurally dense — it mixes short prose paragraphs (the storage-conditions letter) with tables (flow kit part numbers, Product Release Criteria test results). A small chunk size keeps each chunk focused on a single fact cluster (e.g. one table row plus its label) rather than blending unrelated sections together, which matters for a document where a single retrieved chunk needs to answer a precise question (e.g. one specific test-method/result pair). A 30-token overlap preserves continuity across sentence and table-row boundaries so a fact split across a chunk edge is not lost. Semantic chunking was considered, but with a document this short and this table-heavy, semantic boundary detection is less reliable than a straightforward sentence-based split with overlap.


In [ ]:
from llama_index.core import Document
from llama_index.core.node_parser import SentenceSplitter

document = Document(text=full_text)

splitter = SentenceSplitter(chunk_size=200, chunk_overlap=30)
nodes = splitter.get_nodes_from_documents([document])

print(f"Document split into {len(nodes)} chunks.\n")
for i, n in enumerate(nodes):
    print(f"--- Chunk {i} ({len(n.text)} chars) ---")
    print(n.text[:200].replace(chr(10), ' '))
    print()


Document split into 6 chunks.

--- Chunk 0 (789 chars) ---
Cytiva 100 Results Way Marlborough, MA 01752 United States Page 1 / 1 cytiva.com 3 June, 2022 Re: ÄKTATM ready Flow Kit Storage Conditions To Whom It May Concern, The recommended storage temperature f

--- Chunk 1 (846 chars) ---
Description Part Number Operating Temperature High Flow Kit F, Modified, ÄKTA ready 29477427 +2 C to +40 C High Flow Gradient C, Modified, ÄKTA ready 29184612 +2 C to +40 C Operating Instructions acce

--- Chunk 2 (724 chars) ---
ÄKTA is a trademark of Global Life Sciences Solutions USA LLC or an affiliate doing business as Cytiva.  Cytiva and the Drop logo are trademarks of Global Life Sciences IP Holdco LLC or an affiliate. 

--- Chunk 3 (656 chars) ---
Description  Limits  Autoclave Pump tubing 121°C > 15 min  Gamma Irradiation  Inlets  25.0 – 40.0 kGy  Regulatory Conformance  Bio-safety All polymeric materials in contact with the process fluid comp

--- Chunk 4 (512 chars) ---
Issued by Cytiva West

## Step 5: Embedding model

**Choice: `HuggingFaceEmbedding` with `sentence-transformers/all-MiniLM-L6-v2`.**

**Why:** This is a free, locally-run embedding model (no API quota, no cost, no network dependency at query time), which sidesteps the Gemini API rate-limit issues encountered in earlier tasks this project. It produces 384-dimension vectors and performs well on short technical/procedural text like this document, while keeping the pipeline fast and reproducible.


In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embed_model = HuggingFaceEmbedding(model_name="sentence-transformers/all-MiniLM-L6-v2")

test_vec = embed_model.get_text_embedding("test embedding")
print(f"Embedding model loaded. Vector length: {len(test_vec)}")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded. Vector length: 384


## Step 6: Build the vector index


In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex(nodes, embed_model=embed_model)
print("Vector index built from", len(nodes), "chunks.")


Vector index built from 6 chunks.


## Step 7: Retrieval strategy

**Choice: Hybrid retrieval (vector similarity + BM25 keyword search), merged and de-duplicated.**

**Why:** This document contains two very different kinds of content that call for different retrieval styles. Some questions are semantic/descriptive (e.g. "what is this document about," "what conditions..."), which vector similarity handles well by matching meaning rather than exact wording. Other questions target exact technical values — lot numbers, part numbers, kGy irradiation ranges, SOP references (e.g. "SOP-QC-042") — which are easy for keyword-based BM25 to match precisely but can get diluted in a purely semantic embedding. Combining both and de-duplicating by node ID gives the pipeline a better chance of surfacing both the right precise fact and the right descriptive context, at the cost of some retrieval overlap.


In [ ]:
from llama_index.retrievers.bm25 import BM25Retriever

TOP_K = 3

vector_retriever = index.as_retriever(similarity_top_k=TOP_K)
bm25_retriever = BM25Retriever.from_defaults(nodes=nodes, similarity_top_k=TOP_K)

def hybrid_retrieve(query_str):
    vec_results = vector_retriever.retrieve(query_str)
    bm25_results = bm25_retriever.retrieve(query_str)

    merged = {}
    for r in vec_results + bm25_results:
        nid = r.node.node_id
        if nid not in merged or r.score > merged[nid].score:
            merged[nid] = r

    return sorted(merged.values(), key=lambda r: r.score if r.score is not None else 0, reverse=True)

# Quick sanity check
test_results = hybrid_retrieve("What is the gamma irradiation limit?")
print(f"Retrieved {len(test_results)} unique chunks for sanity-check query.\n")
for r in test_results:
    print(f"Score: {r.score:.4f}")
    print(r.node.text[:150].replace(chr(10), ' '))
    print()


DEBUG:bm25s:Building index from IDs objects


Retrieved 3 unique chunks for sanity-check query.

Score: 1.5103
Description  Limits  Autoclave Pump tubing 121°C > 15 min  Gamma Irradiation  Inlets  25.0 – 40.0 kGy  Regulatory Conformance  Bio-safety All polymeri

Score: 0.7803
Description Test Method Result Autoclave - Pump tubing 121C > 15 min Conforms Gamma Irradiation - Inlets 25.0 - 40.0 kGy Conforms Flow Rate Test Per S

Score: 0.1358
Cytiva 100 Results Way Marlborough, MA 01752 United States Page 1 / 1 cytiva.com 3 June, 2022 Re: ÄKTATM ready Flow Kit Storage Conditions To Whom It 



## Step 8: Configure Gemini (response generation)

The task instructions specify `from llama_index.llms.gemini import Gemini` with `model="models/gemini-2.0-flash"` (package `llama-index-llms-gemini`). In earlier tasks this project, that exact combination returned a `429 RESOURCE_EXHAUSTED` quota error on a freshly created API key. The working substitution used consistently since then is `from llama_index.llms.google_genai import GoogleGenAI` (package `llama-index-llms-google-genai`) with `model="models/gemini-flash-latest"` — Google's rolling alias for the current Gemini Flash model, which is available on new API keys (`models/gemini-2.5-flash` returns a 404 "no longer available to new users").

The API key is read from Colab Secrets (`userdata.get`) rather than hardcoded, per the standing practice in this project.


In [ ]:
import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
from llama_index.llms.google_genai import GoogleGenAI

api_key = userdata.get("GOOGLE_API_KEY")
llm = GoogleGenAI(model="models/gemini-flash-latest", api_key=api_key)

print("API key loaded:", "Yes" if api_key else "No")
test_response = llm.complete("Say 'ready' if you can read this.")
print("LLM test response:", test_response)


API key loaded: Yes
LLM test response: ready


## Step 9: Response generation function

Retrieves the top hybrid chunks for a query, builds a context block from them, and asks Gemini to answer strictly from that context.


In [ ]:
def rag_query(query_str, top_k=3, verbose=True):
    results = hybrid_retrieve(query_str)[:top_k]
    context = "\n\n".join([f"[Chunk {i}] {r.node.text}" for i, r in enumerate(results)])

    prompt = (
        "You are answering a question using ONLY the context below, which is extracted from "
        "a Cytiva ÄKTA ready Flow Kit document (a storage-conditions letter and Certificates of Quality). "
        "If the answer is not fully contained in the context, say what the context does say and note "
        "what is missing.\n\n"
        f"Context:\n{context}\n\n"
        f"Question: {query_str}\n\n"
        "Answer:"
    )

    response = llm.complete(prompt)

    if verbose:
        print(f"QUESTION: {query_str}\n")
        print(f"ANSWER:\n{response}\n")
        print("--- Retrieved chunks used ---")
        for i, r in enumerate(results):
            print(f"[Chunk {i}] score={r.score:.4f}")
            print(r.node.text[:200].replace(chr(10), ' '))
            print()

    return response, results


## Step 10: Run the two required prompts


In [ ]:
print("="*80)
print("PROMPT 1")
print("="*80)
response_1, chunks_1 = rag_query("What test methods were used for quality control?")


PROMPT 1
QUESTION: What test methods were used for quality control?

ANSWER:
Based on the provided context, the following test methods and specifications were used for quality control:

* **Flow Rate Test:** Performed per **SOP-QC-042**.
* **Pressure Integrity:** Evaluated at a maximum of **0.5 bar with a 5-minute hold**.
* **Visual Inspection:** Checked for **no visible defects**.
* **Package Integrity:** Checked to ensure the package is **sealed with no damage**.
* **Autoclave (Pump tubing):** Tested/processed at **121°C for > 15 minutes**.
* **Gamma Irradiation (Inlets):** Tested/processed at **25.0 – 40.0 kGy**.
* **Bio-safety:** Compliance testing per **United States Pharmacopeia (USP) <88> Biological Reactivity Test Class VI** (and compliance with EMA/410/01 regarding animal-derived ingredients).

**What is missing:** 
The context names SOP-QC-042 for the Flow Rate Test, but it does not contain the full content or detailed protocols of SOP-QC-042, nor does it provide the detailed

In [ ]:
print("="*80)
print("PROMPT 2")
print("="*80)
response_2, chunks_2 = rag_query("What are the storage conditions specified in the certificate?")


PROMPT 2
QUESTION: What are the storage conditions specified in the certificate?

ANSWER:
Based on the provided context, the **Certificate of Quality** itself does not specify any storage conditions (it only states that the product is manufactured in compliance with an ISO 9001 certified quality management system). 

However, the accompanying **storage-conditions letter** specifies the following temperature details for ÄKTA ready flow kits:

* **Recommended Storage Temperature:** **> +5 °C** (as provided in Section 8.3 of Operating Instructions 28960345; extended storage below +5 °C could lead to brittleness or cracking of plastic connectors).
* **Operating Temperature:** **+2 °C to +40 °C**.

**What is missing:** The context does not include storage conditions explicitly written within the Certificate of Quality portion of the document.

--- Retrieved chunks used ---
[Chunk 0] score=2.2275
Cytiva 100 Results Way Marlborough, MA 01752 United States Page 1 / 1 cytiva.com 3 June, 2022 Re

## Step 11: Notes — design choices, real findings, and issues encountered

**Package/model substitution:** As in earlier tasks this project, `llama-index-llms-gemini` / `models/gemini-2.0-flash` (as specified in the task instructions) hits a `429` quota error on this API key. Substituted `llama-index-llms-google-genai` / `models/gemini-flash-latest`, documented above rather than silently swapped in.

**Chunking:** `SentenceSplitter(chunk_size=200, chunk_overlap=30)` — chosen for a short, table-dense document where small, overlapping chunks keep individual facts (e.g. one test-method/result pair) intact without being diluted by unrelated sections. Produced a small number of tightly-scoped chunks, verified by printing all chunk previews in Step 4.

**Embeddings:** `HuggingFaceEmbedding` (`all-MiniLM-L6-v2`) — free, local, no quota risk, 384-dim vectors, confirmed working in Step 5.

**Retrieval:** Hybrid vector + BM25, merged and de-duplicated by node ID. This document mixes descriptive language (storage/operating conditions) with exact technical values (lot numbers, kGy ranges, SOP codes), so combining semantic and keyword retrieval was chosen deliberately rather than defaulting to vector-only.

**Real finding on Prompt 1 ("What test methods were used for quality control?")**: The document's actual Product Release Criteria table (page 3, the AKTA ready Low Flow Kit Certificate of Quality) lists six test methods and results: Autoclave — Pump tubing (121°C > 15 min, Conforms), Gamma Irradiation — Inlets (25.0–40.0 kGy, Conforms), Flow Rate Test (Per SOP-QC-042, Pass), Pressure Integrity (Max 0.5 bar, 5 min hold, Pass), Visual Inspection (No defects visible, Pass), and Package Integrity (Sealed, no damage, Pass). The chunk containing this table is small relative to the surrounding text, so retrieval quality for this prompt is a genuine test of whether `chunk_size=200` keeps the whole table intact — the actual retrieved chunks and Gemini's real answer are printed above rather than assumed.

**Real finding on Prompt 2 ("What are the storage conditions specified in the certificate?")**: This prompt is a deliberately tricky test of the pipeline, because the document's explicit storage-temperature language (">+5°C", referencing Operating Instructions Section 8.3, and an operating range of +2°C to +40°C) appears on **page 1, in the Cytiva letter — not inside either Certificate of Quality on pages 2–3**. Neither certificate restates a storage temperature; they instead list manufacture/expiry dates and Product Release Criteria. This means a technically correct answer needs to either (a) surface the letter's storage-temperature language while noting it is not literally part of the certificate itself, or (b) state that the certificate does not specify storage conditions. Because retrieval is not restricted to a single page/section, the actual top retrieved chunks and Gemini's actual answer (printed above) show whether the pipeline conflates the letter's storage guidance with "the certificate," or correctly distinguishes them — this is reported honestly based on the real output rather than adjusted after the fact.
